# FlashWorld → room `.ply` + preview video (Kaggle)

Headless generation of one explorable Gaussian-splat room from a **single room photo**, for the Tile-Visualizer demo.

## Before Run All

In the Kaggle notebook sidebar (**Settings** / accelerator panel):

1. **Accelerator** → GPU (`T4` or `P100`, 16 GB)
2. **Internet** → **On** (required to clone FlashWorld + download weights)
3. Upload your room photo to this notebook (large, flat walls work best for later tile panels)

No Hugging Face token or other credentials are required — both weight repos used below are public/ungated.

## Verified tooling source

Checked live on **2026-07-12** (do not trust stale install commands; re-check if something fails):

| Source | URL |
|--------|-----|
| GitHub README + CLI | https://github.com/imlixinyang/FlashWorld |
| Hugging Face demo | https://huggingface.co/spaces/imlixinyang/FlashWorld-Demo-Spark |
| Checkpoint | `imlixinyang/FlashWorld` (`model.ckpt`, ungated) |
| Base diffusion weights | `Wan-AI/Wan2.2-TI2V-5B-Diffusers` (ungated) |

**Install (from README, torch skipped — Kaggle already has it):** pinned `gsplat@32f2a54…`, `diffusers@447e832…`, `spz@a4fc69e…`.

**CLI (from README / `cli.py`):** `python cli.py --input_dir … --output_dir … --video --ply` plus shared offload flags:

- `--offload_t5`
- `--offload_transformer_during_vae`
- `--offload_vae` ← required to stay under ~10 GB on a 16 GB card (README: ~9 GB / ~10 min on A800; expect slower on T4/P100)

**Input format:** CLI reads a directory of JSON jobs (`image_prompt`, `text_prompt`, `cameras`, `resolution`, `image_index`). `image_prompt` may be a **file path** (not only base64).


## 1. Sanity-check GPU


In [ ]:
import subprocess
import sys

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=False)

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError(
        "No CUDA GPU. Set Accelerator to GPU (T4/P100) in Kaggle Settings and restart the session."
    )


## 2. Install FlashWorld dependencies

Skips reinstalling `torch` / `torchvision` (preinstalled on Kaggle). Installs the **pinned** git commits from the FlashWorld README so the Wan-compatible `diffusers` fork matches the checkpoint.

First run also pulls large public weights (~tens of GB total). Keep **Internet** on and watch free disk if downloads fail.


In [ ]:
import os
import subprocess
import sys

# T4 = 7.5, P100 = 6.0 — helps gsplat CUDA extension build on Kaggle
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "6.0;7.5")

def pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# README "install packages" (minus torch/torchvision)
pip_install(
    "triton",
    "transformers",
    "omegaconf",
    "ninja",
    "numpy",
    "jaxtyping",
    "rich",
    "einops",
    "moviepy==1.0.3",
    "accelerate",
    "opencv-python",
    "av",
    "plyfile",
    "ftfy",
    "pandas",
    "uvicorn",
    "nanobind",
    "imageio",
    "imageio-ffmpeg",
    "tqdm",
    "huggingface_hub",
)

# README pinned VCS deps (exact commits verified 2026-07-12)
pip_install(
    "git+https://github.com/nerfstudio-project/gsplat.git@32f2a54d21c7ecb135320bb02b136b7407ae5712",
)
pip_install(
    "git+https://github.com/huggingface/diffusers.git@447e8322f76efea55d4769cd67c372edbf0715b8",
)
pip_install(
    "git+https://github.com/nianticlabs/spz.git@a4fc69e7948c7152e807e6501d73ddc9c149ce37",
)

print("Dependencies installed.")


## 3. Clone FlashWorld


In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/kaggle/working/FlashWorld")
REPO_URL = "https://github.com/imlixinyang/FlashWorld.git"

if not (REPO_DIR / "cli.py").exists():
    if REPO_DIR.exists():
        # Incomplete clone — start clean
        subprocess.check_call(["rm", "-rf", str(REPO_DIR)])
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    print(f"Already present: {REPO_DIR}")

assert (REPO_DIR / "cli.py").is_file()
print("FlashWorld ready at", REPO_DIR)


## 4. ✏️ SET YOUR ROOM PHOTO PATH (edit this cell)

Upload a room photo with **large, flat walls**, then set `ROOM_PHOTO_PATH` below.

Typical Kaggle locations after upload:

- `/kaggle/working/my_room.jpg` (uploaded into the notebook output area)
- `/kaggle/input/<your-dataset-or-upload>/my_room.jpg`


In [ ]:
from pathlib import Path

# =============================================================================
# EDIT ME — path to your uploaded room photo (jpg/png)
# =============================================================================
ROOM_PHOTO_PATH = "/kaggle/working/room.jpg"

# Optional caption; empty string is fine for image-only conditioning
TEXT_PROMPT = "An indoor room with large flat walls and a clear floor, suitable for tile visualization."

# Generation resolution: [n_frames, height, width]
# Examples / README default often use 24×480×704. Start slightly lighter for 16 GB cards:
RESOLUTION = [16, 480, 704]

# If OOM persists after all offload flags, try e.g. [12, 384, 576]
# =============================================================================

ROOM_PHOTO_PATH = str(Path(ROOM_PHOTO_PATH).expanduser())
assert Path(ROOM_PHOTO_PATH).is_file(), (
    f"Room photo not found: {ROOM_PHOTO_PATH}\n"
    "Upload the image in the Kaggle UI, then update ROOM_PHOTO_PATH."
)
print("Using room photo:", ROOM_PHOTO_PATH)
print("RESOLUTION:", RESOLUTION)


## 5. Build CLI input JSON

FlashWorld’s CLI does **not** take a raw image flag — it reads JSON jobs. We reuse the **camera trajectory from the indoor `examples/5.json`** (sound-equipment room) and swap in your photo path. That keeps a real multi-view path without hand-tuning quaternions.

For the absolute best path, the upstream README recommends drafting cameras in the [web / HF demo](https://huggingface.co/spaces/imlixinyang/FlashWorld-Demo-Spark) and pasting that JSON here instead.


In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np

REPO_DIR = Path("/kaggle/working/FlashWorld")
INPUT_DIR = Path("/kaggle/working/fw_input")
OUTPUT_DIR = Path("/kaggle/working/fw_output")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Copy photo next to the JSON so the job is self-contained and path-stable
photo_src = Path(ROOM_PHOTO_PATH)
photo_dst = INPUT_DIR / f"room{photo_src.suffix.lower()}"
shutil.copy2(photo_src, photo_dst)

template_path = REPO_DIR / "examples" / "5.json"
with template_path.open("r", encoding="utf-8") as f:
    job = json.load(f)

# Drop the huge embedded example image; point at our file path instead
job["image_prompt"] = str(photo_dst)
job["text_prompt"] = TEXT_PROMPT
job["resolution"] = list(RESOLUTION)

# Keep template cameras + image_index, but resample if we changed n_frames
n_frames = int(RESOLUTION[0])
n_cams = len(job["cameras"])
if n_cams != n_frames:
    idxs = np.linspace(0, n_cams - 1, n_frames)
    idxs = np.round(idxs).astype(int)
    job["cameras"] = [job["cameras"][int(i)] for i in idxs]
    job["image_index"] = int(n_frames // 2)
else:
    job["image_index"] = int(min(job.get("image_index", n_frames // 2), n_frames - 1))

job_path = INPUT_DIR / "room.json"
with job_path.open("w", encoding="utf-8") as f:
    json.dump(job, f)

print("Wrote", job_path)
print("  image_prompt:", job["image_prompt"])
print("  resolution:", job["resolution"])
print("  image_index:", job["image_index"])
print("  n cameras:", len(job["cameras"]))


## 6. Run FlashWorld CLI (all offload flags)

Expect **tens of minutes** on a T4/P100 with `--offload_vae` (README: ~10 min / ~9 GB on A800 with the same flag set). Prefer **Save Version → Save & Run All** (or background execution) so a browser disconnect does not kill the job.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/FlashWorld")
INPUT_DIR = Path("/kaggle/working/fw_input")
OUTPUT_DIR = Path("/kaggle/working/fw_output")

os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "6.0;7.5")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    sys.executable,
    "cli.py",
    "--input_dir", str(INPUT_DIR),
    "--output_dir", str(OUTPUT_DIR),
    "--video",
    "--ply",
    # Keep peak VRAM under ~10 GB on 16 GB Kaggle GPUs (README table)
    "--offload_t5",
    "--offload_transformer_during_vae",
    "--offload_vae",
]

print("Running:", " ".join(cmd))
print("CWD:", REPO_DIR)
subprocess.check_call(cmd, cwd=str(REPO_DIR))
print("CLI finished.")


## 7. Collect downloadable artifacts into `/kaggle/working`


In [ ]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/fw_output")
WORKING = Path("/kaggle/working")

# CLI writes per-JSON subdirs: fw_output/room/{gaussians.ply, video.mp4, ...}
candidates = list(OUTPUT_DIR.rglob("gaussians.ply"))
assert candidates, f"No gaussians.ply under {OUTPUT_DIR}. Check the CLI cell logs."

src_ply = candidates[0]
src_dir = src_ply.parent
src_video = src_dir / "video.mp4"

dst_ply = WORKING / "room.ply"
dst_video = WORKING / "room_preview.mp4"

shutil.copy2(src_ply, dst_ply)
print("Wrote", dst_ply, f"({dst_ply.stat().st_size / 1e6:.1f} MB)")

if src_video.is_file():
    shutil.copy2(src_video, dst_video)
    print("Wrote", dst_video, f"({dst_video.stat().st_size / 1e6:.1f} MB)")
else:
    print("No video.mp4 found next to PLY (generation may have skipped video).")

src_spz = src_dir / "gaussians.spz"
if src_spz.is_file():
    dst_spz = WORKING / "room.spz"
    shutil.copy2(src_spz, dst_spz)
    print("Wrote", dst_spz)

print("\nDownload from the Kaggle Output /working panel:")
print("  - room.ply          → drop into Tile-Visualizer public/scene/room.ply")
print("  - room_preview.mp4  → quick visual QA")


## Troubleshooting

### CUDA OOM / “out of memory”
1. Confirm the CLI cell includes **all three** flags: `--offload_t5`, `--offload_transformer_during_vae`, `--offload_vae`.
2. Lower `RESOLUTION` in the edit cell, e.g. `[12, 384, 576]` or `[8, 384, 512]`, then rebuild the JSON and re-run the CLI.
3. Restart the Kaggle session to clear fragmented VRAM before retrying.
4. Do not remove `--offload_vae` on a 16 GB card — that flag is what drops usage below ~10 GB (README).

### Long runtime / browser disconnect
With `--offload_vae`, generation is slow (order of **10+ minutes** even on A800). Use Kaggle **Save & Run All** / commit-and-run so the job continues in the background; download artifacts from the completed version’s output.

### Install / download failures
- Internet must be **On**.
- Disk: first run downloads `imlixinyang/FlashWorld` (`model.ckpt`) and Wan 2.2 TI2V components — several tens of GB. Free space or clear `/root/.cache/huggingface` if pulls fail.
- `gsplat` build errors: ensure GPU accelerator is enabled before the install cell (CUDA headers/arch), and that `TORCH_CUDA_ARCH_LIST` includes `6.0;7.5`.

### Fallback (demo must not depend on live generation)
If Kaggle still OOMs after reasonable retries, or weekly GPU quota is exhausted, switch to a **local Depth Anything V2** depth-displaced mesh for a stand-in room. Keep any successful FlashWorld `.ply` as the preferred asset; the Tile-Visualizer demo always loads a **pre-baked** scene from `public/scene/` and never calls a live generator.
